# Runtime Batch Benchmark (qubit-TransmonCross-Hamiltonian_params)
## Hamiltonian to Quantum Metal to Hamiltonian

Inverse plus surrogate and surrogate-only inference timing.

This notebook benchmarks time per sample versus batch number on CPU and GPU. The plotting style follows Figure 8 from the qCHeff paper: tick-style axes, log runtime axis, slategray baseline, CPU blue, and GPU green.

## Configuration

In [ ]:
## the parameter file has the path names for this experiment
## start there if you want to change the setup

from parameters_surrogate_defined_loss import *

## Library

In [ ]:
import os, gc, json, time, platform, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import load_model

try:
    for gpu in tf.config.list_physical_devices('GPU'):
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception as e:
    print('Could not set GPU memory growth:', repr(e))

In [ ]:
## combined path matches chosen_path in ml_22
## surrogate path is the frozen model used inside that combined model
encoding = 'surrogate_defined_loss'
COMBINED_MODEL_PATH = Path(MODEL_DIR) / f'best_keras_model_{encoding}.keras'
SURROGATE_MODEL_PATH = Path(MODEL_DIR) / 'best_keras_model_model2_surrogate.keras'

RUNTIME_DIR = Path(RESULTS_DIR) / 'runtime'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
Path(PLOTS_DIR).mkdir(parents=True, exist_ok=True)

print(f'Combined inverse + surrogate model: {COMBINED_MODEL_PATH}')
print(f'Surrogate-only model:              {SURROGATE_MODEL_PATH}')
print(f'Runtime results dir:               {RUNTIME_DIR}')
print(f'Plots dir:                         {PLOTS_DIR}')

for model_path in [COMBINED_MODEL_PATH, SURROGATE_MODEL_PATH]:
    if not model_path.exists():
        raise FileNotFoundError(f'Missing model file: {model_path}')

## Dataset

### Load

In [ ]:
def _load_npy(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f'Missing data file: {path}. If this is a fresh clone, unzip the supplemental data into this experiment folder first.'
        )
    return np.load(path, allow_pickle=True).astype('float32')

## combined model input is scaled Hamiltonian from ml_22
X_inverse_test = _load_npy(Path(DATA_DIR) / 'npy' / 'x_test_one_hot_encoding_augmented.npy')

## surrogate model input is scaled Quantum Metal params from ml_12/ml_21
X_surrogate_test = _load_npy(Path(DATA_DIR) / 'npy' / 'y_test_linear_encoding_scaled.npy')

with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
    Hamiltonian_column_names = f.read().splitlines()
qiskit_param_names = np.load(str(Path(METADATA_DIR) / 'y_columns.npy'), allow_pickle=True).astype(str).tolist()

print(f'Combined model benchmark input: {X_inverse_test.shape}')
print(f'Surrogate-only benchmark input: {X_surrogate_test.shape}')
print(f'Hamiltonian columns: {Hamiltonian_column_names}')
print(f'Quantum Metal columns: {qiskit_param_names}')

### Define conversion layer

In [ ]:
## must define this class before loading the saved combined model
class ScalerConversionLayer(tf.keras.layers.Layer):
    def __init__(self, scale_a, scale_b, **kwargs):
        kwargs.setdefault('trainable', False)
        super().__init__(**kwargs)
        self._scale_a = tf.constant(scale_a, dtype=tf.float32)
        self._scale_b = tf.constant(scale_b, dtype=tf.float32)
        self._cfg = dict(
            scale_a=list(scale_a) if hasattr(scale_a, '__iter__') else scale_a,
            scale_b=list(scale_b) if hasattr(scale_b, '__iter__') else scale_b,
        )

    def call(self, inputs):
        a = tf.cast(self._scale_a, inputs.dtype)
        b = tf.cast(self._scale_b, inputs.dtype)
        return inputs * a + b

    def get_config(self):
        config = super().get_config()
        config.update(self._cfg)
        return config

## Benchmark Setup

In [ ]:
## change these if you want a longer or shorter benchmark
BATCH_NUMBERS = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]
N_WARMUP = 20
N_REPEAT = 50
RUN_CPU_BENCHMARK = True
RUN_GPU_BENCHMARK = True

## placeholder for the third surrogate-only histogram bar
## update this if a newer measured Ansys time should be used
ANSYS_SECONDS_PER_SAMPLE = 2.0 * 60.0

QCHEFF_COLORS = {
    'Ansys': 'slategray',
    'CPU': '#127cc1',
    'GPU': '#76b900',
}
QCHEFF_MARKERS = {
    'Ansys': 's',
    'CPU': 'v',
    'GPU': '^',
}

GPU_DEVICES = tf.config.list_physical_devices('GPU')
GPU_AVAILABLE = len(GPU_DEVICES) > 0
print('CPU benchmark:', RUN_CPU_BENCHMARK)
print('GPU benchmark:', RUN_GPU_BENCHMARK, '| visible GPUs:', GPU_DEVICES)

In [ ]:
def make_batch(X, batch_number):
    ## tile the test set when a requested batch is larger than the held-out set
    X = np.asarray(X, dtype=np.float32)
    if len(X) >= batch_number:
        return X[:batch_number]
    reps = int(np.ceil(batch_number / len(X)))
    return np.tile(X, (reps, 1))[:batch_number]


def sync_tensorflow_outputs(outputs):
    ## force GPU work to finish before stopping the timer
    if isinstance(outputs, (list, tuple)):
        for output in outputs:
            sync_tensorflow_outputs(output)
    elif isinstance(outputs, dict):
        for output in outputs.values():
            sync_tensorflow_outputs(output)
    elif tf.is_tensor(outputs):
        _ = outputs.numpy()
    else:
        _ = np.asarray(outputs)


def call_model_once(model, x_tensor):
    outputs = model(x_tensor, training=False)
    sync_tensorflow_outputs(outputs)


def benchmark_model_path(model_path, X, benchmark_name, device_label, device_name, custom_objects=None):
    rows = []
    custom_objects = custom_objects or {}

    if device_label == 'GPU' and not GPU_AVAILABLE:
        print(f'Skipping {benchmark_name} on GPU because no TensorFlow GPU is visible.')
        return rows

    print(f'Loading {benchmark_name} on {device_label}: {model_path}')
    tf.keras.backend.clear_session()
    gc.collect()

    try:
        with tf.device(device_name):
            model = load_model(model_path, compile=False, custom_objects=custom_objects)
    except Exception as e:
        rows.append({
            'benchmark': benchmark_name,
            'device': device_label,
            'batch_number': np.nan,
            'repeat': np.nan,
            'status': 'load_error',
            'error': repr(e),
        })
        print(f'Could not load {benchmark_name} on {device_label}: {repr(e)}')
        return rows

    for batch_number in BATCH_NUMBERS:
        X_batch = make_batch(X, batch_number)
        try:
            with tf.device(device_name):
                x_tensor = tf.convert_to_tensor(X_batch, dtype=tf.float32)
                actual_tensor_device = x_tensor.device

                ## warmup to trigger tracing and kernel initialization
                for _ in range(N_WARMUP):
                    call_model_once(model, x_tensor)

                for repeat in range(N_REPEAT):
                    t0 = time.perf_counter()
                    call_model_once(model, x_tensor)
                    total_seconds = time.perf_counter() - t0

                    rows.append({
                        'benchmark': benchmark_name,
                        'device': device_label,
                        'device_name': device_name,
                        'actual_tensor_device': actual_tensor_device,
                        'batch_number': int(batch_number),
                        'repeat': int(repeat),
                        'total_seconds': float(total_seconds),
                        'time_per_sample_seconds': float(total_seconds / batch_number),
                        'status': 'ok',
                        'error': '',
                    })
        except Exception as e:
            rows.append({
                'benchmark': benchmark_name,
                'device': device_label,
                'device_name': device_name,
                'batch_number': int(batch_number),
                'repeat': np.nan,
                'status': 'benchmark_error',
                'error': repr(e),
            })
            print(f'Benchmark error for {benchmark_name} on {device_label}, batch {batch_number}: {repr(e)}')

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return rows

## Run Benchmarks

In [ ]:
all_timing_rows = []

if RUN_CPU_BENCHMARK:
    all_timing_rows.extend(benchmark_model_path(
        COMBINED_MODEL_PATH,
        X_inverse_test,
        benchmark_name='inverse+surrogate',
        device_label='CPU',
        device_name='/CPU:0',
        custom_objects={'ScalerConversionLayer': ScalerConversionLayer},
    ))
    all_timing_rows.extend(benchmark_model_path(
        SURROGATE_MODEL_PATH,
        X_surrogate_test,
        benchmark_name='surrogate-only',
        device_label='CPU',
        device_name='/CPU:0',
    ))

if RUN_GPU_BENCHMARK:
    all_timing_rows.extend(benchmark_model_path(
        COMBINED_MODEL_PATH,
        X_inverse_test,
        benchmark_name='inverse+surrogate',
        device_label='GPU',
        device_name='/GPU:0',
        custom_objects={'ScalerConversionLayer': ScalerConversionLayer},
    ))
    all_timing_rows.extend(benchmark_model_path(
        SURROGATE_MODEL_PATH,
        X_surrogate_test,
        benchmark_name='surrogate-only',
        device_label='GPU',
        device_name='/GPU:0',
    ))

timing_raw_df = pd.DataFrame(all_timing_rows)
timing_raw_df.head()

### Save

In [ ]:
raw_path = RUNTIME_DIR / 'ml_30_runtime_batch_benchmark_raw.csv'
summary_path = RUNTIME_DIR / 'ml_30_runtime_batch_benchmark_summary.csv'
metadata_path = RUNTIME_DIR / 'ml_30_runtime_batch_benchmark_metadata.json'

if timing_raw_df.empty:
    raise RuntimeError('No timing rows were produced. Check benchmark settings and visible devices.')

timing_raw_df.to_csv(raw_path, index=False)

ok_df = timing_raw_df[timing_raw_df['status'] == 'ok'].copy()
if ok_df.empty:
    raise RuntimeError('No successful timing rows were produced. Check timing_raw_df for errors.')

summary_df = (
    ok_df
    .groupby(['benchmark', 'device', 'batch_number'], as_index=False)
    .agg(
        mean_total_seconds=('total_seconds', 'mean'),
        std_total_seconds=('total_seconds', 'std'),
        mean_time_per_sample_seconds=('time_per_sample_seconds', 'mean'),
        std_time_per_sample_seconds=('time_per_sample_seconds', 'std'),
        repeats=('time_per_sample_seconds', 'count'),
    )
)
summary_df['mean_time_per_sample_ms'] = summary_df['mean_time_per_sample_seconds'] * 1000.0
summary_df['std_time_per_sample_ms'] = summary_df['std_time_per_sample_seconds'] * 1000.0
summary_df.to_csv(summary_path, index=False)

metadata = {
    'created_by': 'ml_30_runtime_batch_benchmark.ipynb',
    'system': f'{platform.system()} {platform.release()}',
    'machine': platform.machine(),
    'processor': platform.processor(),
    'python': platform.python_version(),
    'tensorflow': tf.__version__,
    'gpu_available': GPU_AVAILABLE,
    'gpu_devices': [str(g) for g in GPU_DEVICES],
    'batch_numbers': BATCH_NUMBERS,
    'n_warmup': N_WARMUP,
    'n_repeat': N_REPEAT,
    'combined_model_path': str(COMBINED_MODEL_PATH),
    'surrogate_model_path': str(SURROGATE_MODEL_PATH),
    'ansys_seconds_per_sample_placeholder': ANSYS_SECONDS_PER_SAMPLE,
}
with metadata_path.open('w') as f:
    json.dump(metadata, f, indent=2)

print(f'Saved raw timing data -> {raw_path}')
print(f'Saved timing summary -> {summary_path}')
print(f'Saved timing metadata -> {metadata_path}')
summary_df

In [ ]:
## save split CSV files too, since these are the two datasets plotted below
for benchmark_name in ['inverse+surrogate', 'surrogate-only']:
    safe_name = benchmark_name.replace('+', '_plus_').replace('-', '_')
    out_path = RUNTIME_DIR / f'ml_30_{safe_name}_batch_timing_summary.csv'
    summary_df[summary_df['benchmark'] == benchmark_name].to_csv(out_path, index=False)
    print(f'Saved {benchmark_name} summary -> {out_path}')

## Plot Helpers

In [ ]:
def add_ansys_placeholder(plot_df):
    batches = sorted(plot_df['batch_number'].dropna().unique())
    ansys_df = pd.DataFrame({
        'benchmark': 'surrogate-only',
        'device': 'Ansys',
        'batch_number': batches,
        'mean_time_per_sample_seconds': ANSYS_SECONDS_PER_SAMPLE,
        'std_time_per_sample_seconds': 0.0,
        'mean_time_per_sample_ms': ANSYS_SECONDS_PER_SAMPLE * 1000.0,
        'std_time_per_sample_ms': 0.0,
        'repeats': 1,
    })
    return pd.concat([plot_df, ansys_df], ignore_index=True)


def save_figure(fig, stem):
    pdf_path = Path(PLOTS_DIR) / f'{stem}.pdf'
    png_path = Path(PLOTS_DIR) / f'{stem}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    print(f'Saved plot -> {pdf_path}')
    print(f'Saved plot -> {png_path}')


def plot_figure8_lines(plot_df, title, stem, include_ansys=False):
    if include_ansys:
        plot_df = add_ansys_placeholder(plot_df)

    order = [d for d in ['Ansys', 'CPU', 'GPU'] if d in set(plot_df['device'])]
    with (
        sns.axes_style('ticks'),
        sns.plotting_context('notebook'),
        matplotlib.rc_context({'mathtext.fontset': 'cm'}),
    ):
        fig, ax = plt.subplots(figsize=(6, 4), layout='constrained')
        for device in order:
            sub = plot_df[plot_df['device'] == device].sort_values('batch_number')
            ax.plot(
                sub['batch_number'],
                sub['mean_time_per_sample_seconds'],
                marker=QCHEFF_MARKERS[device],
                markersize=7,
                lw=3,
                alpha=0.9,
                color=QCHEFF_COLORS[device],
                label=device,
            )
        ax.set_xscale('log', base=2)
        ax.set_yscale('log')
        ax.set_xlabel('Batch Number')
        ax.set_ylabel('Time per sample (s)')
        ax.set_title(title)
        ax.set_xticks(BATCH_NUMBERS)
        ax.set_xticklabels([str(b) for b in BATCH_NUMBERS], rotation=45, ha='right')
        ax.grid(axis='y')
        ax.legend(frameon=False)
    save_figure(fig, stem)
    return fig, ax


def plot_figure8_histogram(plot_df, title, stem, include_ansys=False):
    if include_ansys:
        plot_df = add_ansys_placeholder(plot_df)

    batches = sorted(plot_df['batch_number'].dropna().unique())
    order = [d for d in ['Ansys', 'CPU', 'GPU'] if d in set(plot_df['device'])]
    x = np.arange(len(batches))
    width = 0.8 / max(len(order), 1)

    with (
        sns.axes_style('ticks'),
        sns.plotting_context('notebook'),
        matplotlib.rc_context({'mathtext.fontset': 'cm'}),
    ):
        fig, ax = plt.subplots(figsize=(7, 4), layout='constrained')
        for i, device in enumerate(order):
            sub = (
                plot_df[plot_df['device'] == device]
                .set_index('batch_number')
                .reindex(batches)
            )
            offset = (i - (len(order) - 1) / 2.0) * width
            ax.bar(
                x + offset,
                sub['mean_time_per_sample_seconds'],
                width=width,
                color=QCHEFF_COLORS[device],
                alpha=0.85,
                edgecolor='black',
                linewidth=0.6,
                label=device,
            )
        ax.set_yscale('log')
        ax.set_xlabel('Batch Number')
        ax.set_ylabel('Time per sample (s)')
        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels([str(b) for b in batches], rotation=45, ha='right')
        ax.grid(axis='y')
        ax.legend(frameon=False)
    save_figure(fig, stem)
    return fig, ax

## Inverse Plus Surrogate Runtime

In [ ]:
inverse_surrogate_df = summary_df[summary_df['benchmark'] == 'inverse+surrogate'].copy()
inverse_surrogate_df

In [ ]:
fig, ax = plot_figure8_lines(
    inverse_surrogate_df,
    title='Inverse + Surrogate Runtime',
    stem='ml_30_inverse_surrogate_runtime_vs_batch_number',
    include_ansys=False,
)
plt.show()

In [ ]:
fig, ax = plot_figure8_histogram(
    inverse_surrogate_df,
    title='Inverse + Surrogate Runtime Histogram',
    stem='ml_30_inverse_surrogate_runtime_histogram_vs_batch_number',
    include_ansys=False,
)
plt.show()

## Surrogate-Only Runtime

In [ ]:
surrogate_only_df = summary_df[summary_df['benchmark'] == 'surrogate-only'].copy()
surrogate_only_plot_df = add_ansys_placeholder(surrogate_only_df)

surrogate_plot_path = RUNTIME_DIR / 'ml_30_surrogate_only_batch_timing_summary_with_ansys_placeholder.csv'
surrogate_only_plot_df.to_csv(surrogate_plot_path, index=False)
print(f'Saved surrogate plot data with Ansys placeholder -> {surrogate_plot_path}')
surrogate_only_plot_df

In [ ]:
fig, ax = plot_figure8_lines(
    surrogate_only_df,
    title='Surrogate-Only Runtime',
    stem='ml_30_surrogate_only_runtime_vs_batch_number',
    include_ansys=True,
)
plt.show()

In [ ]:
fig, ax = plot_figure8_histogram(
    surrogate_only_df,
    title='Surrogate-Only Runtime Histogram',
    stem='ml_30_surrogate_only_runtime_histogram_vs_batch_number',
    include_ansys=True,
)
plt.show()

## Quick Comparison

In [ ]:
## compact table with the fastest per-sample timing for each benchmark/device
fastest = (
    summary_df
    .sort_values('mean_time_per_sample_seconds')
    .groupby(['benchmark', 'device'], as_index=False)
    .first()[['benchmark', 'device', 'batch_number', 'mean_time_per_sample_seconds', 'mean_time_per_sample_ms']]
)
fastest